## Evaluate a custom Presidio Analyzer using the Presidio Evaluator framework

This notebook demonstrates how to evaluate a Presidio instance using the presidio-evaluator framework. It builds upon [example 4](4_Evaluate_Presidio_Analyzer.ipynb), with changes to the `PresidioAnalyzer` instance to improve detection accuracy. For more information on customizing the Presidio Analyzer, see the [Presidio Analyzer documentation](https://microsoft.github.io/presidio/analyzer/) or this [tutorial](https://microsoft.github.io/presidio/tutorial/).

Steps:
1. Load dataset from file
2. Simple dataset statistics
3. Define the AnalyzerEngine object (and its parameters)
4. Interactive entity mapping with EntityMappingHelper
5. Set up the Evaluator object
6. Run experiment
7. Evaluate results
8. Error analysis

In [89]:
# install presidio evaluator via pip if not yet installed

#!pip install presidio-evaluator
#!pip install "presidio-analyzer[transformers]"

In [90]:
from pathlib import Path
from pprint import pprint
from collections import Counter
from typing import Dict, List
import json
import warnings
warnings.filterwarnings('ignore')

from presidio_evaluator import InputSample
from presidio_evaluator.evaluation import SpanEvaluator, ModelError, Plotter
from presidio_evaluator.models import PresidioAnalyzerWrapper
from presidio_evaluator.experiment_tracking import get_experiment_tracker
from presidio_evaluator.entity_mapping import EntityMappingHelper


from presidio_analyzer import AnalyzerEngine, PatternRecognizer, RecognizerRegistry, Pattern
from presidio_analyzer.nlp_engine import TransformersNlpEngine
from presidio_analyzer.context_aware_enhancers import LemmaContextAwareEnhancer

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

%reload_ext autoreload
%autoreload 2
%matplotlib inline

## 1. Load dataset from file

In [91]:
dataset_name = "synth_dataset_v2.json"
dataset = InputSample.read_dataset_json(Path(Path.cwd().parent, "data", dataset_name))

print(len(dataset))

tokenizing input: 100%|██████████| 1500/1500 [00:06<00:00, 239.46it/s]

1500


This dataset was auto generated. See more info here [Synthetic data generation](1_Generate_data.ipynb).

In [92]:
def get_entity_counts(dataset: List[InputSample]) -> Dict:
    """Return a dictionary with counter per entity type."""
    entity_counter = Counter()
    for sample in dataset:
        for tag in sample.tags:
            entity_counter[tag] += 1
    return entity_counter


## 2. Simple dataset statistics

In [93]:
entity_counts = get_entity_counts(dataset)
print("Count per entity:")
pprint(entity_counts.most_common(), compact=True)

print("\nMin and max number of tokens in dataset: "\
f"Min: {min([len(sample.tokens) for sample in dataset])}, "\
f"Max: {max([len(sample.tokens) for sample in dataset])}")

print(f"Min and max sentence length in dataset: " \
f"Min: {min([len(sample.full_text) for sample in dataset])}, "\
f"Max: {max([len(sample.full_text) for sample in dataset])}")

Count per entity:
[('O', 19626), ('STREET_ADDRESS', 3071), ('PERSON', 1369), ('GPE', 521),
 ('ORGANIZATION', 504), ('PHONE_NUMBER', 350), ('DATE_TIME', 219),
 ('TITLE', 142), ('CREDIT_CARD', 136), ('US_SSN', 80), ('AGE', 74), ('NRP', 55),
 ('ZIP_CODE', 50), ('EMAIL_ADDRESS', 49), ('DOMAIN_NAME', 37),
 ('IP_ADDRESS', 22), ('IBAN_CODE', 21), ('US_DRIVER_LICENSE', 9)]

Min and max number of tokens in dataset: Min: 3, Max: 78
Min and max sentence length in dataset: Min: 9, Max: 407


## 3. Define the AnalyzerEngine object 
In this case, we customize the AnalyzerEngine to use a different NER model, some custom recognizers and the context aware enhancer.

### 3.1 Set up the NlpEngine
The NLP engine is in charge of text processing using spaCy, and named entity recognition using a transformers model

In [94]:
# Define which model to use
from presidio_analyzer.nlp_engine import NerModelConfiguration


models = [{"lang_code": "en", "model_name": {
    "spacy": "en_core_web_sm",  # use a small spaCy model for lemmas, tokens etc.
    "transformers": "StanfordAIMI/stanford-deidentifier-base"
    }
}]

mapping = dict(
    VENDOR="ORGANIZATION",
  	DATE="DATE_TIME",
  	HCW="PERSON",
  	HOSPITAL="ORGANIZATION",
  	ID="ID",
  	PATIENT="PERSON",
  	PHONE="PHONE_NUMBER"
)
# Reduce ID entity scores to prevent false detection of CREDIT_CARD and IBAN_CODE as ID
# Multiplier of 0.3 ensures pattern recognizers (CREDIT_CARD, IBAN) win over generic ID predictions
ner_config = NerModelConfiguration(
    model_to_presidio_entity_mapping=mapping,
	low_score_entity_names=["ID"], 
    low_confidence_score_multiplier=0.3
)

# Create the NLP engine without entity mapping (will be handled by EntityMappingHelper)
nlp_engine = TransformersNlpEngine(models=models, ner_model_configuration=ner_config)
nlp_engine.load()

Device set to use cpu


### 3.2 Set up the relevant recognizers
Add and remove recognizers to fit the dataset in hand. 
Adding simple titles and zip code recognizers, another deny list for things that aren't considered PII but labeled as such,
and removing all the recognizers that don't map to entities in our dataset.

In [95]:
# Add a simple deny-list recognizer for titles
def get_titles_recognizer():
    titles_recognizer = PatternRecognizer(deny_list=["Mr.", "Mrs.", "Ms.", "Miss", "Dr.", "Prof.", "manager"],
                                          supported_entity="TITLE", name="TitlesRecognizer")
    return titles_recognizer

def get_location_deny_list():
    location_deny_list = PatternRecognizer(deny_list=["APO", "PSC", "AA", "Cyprus (Greek)", 
                                                      "ul", "AE", "DPO", "AP", "nan"],
                                          supported_entity="LOCATION", name="LocationDenylist")
    return location_deny_list

# Add a simple regex to identify ages
def get_age_recognizer():
    weak_regex= r"\b(110|[1-9]?[0-9])\b"
    age_pattern = Pattern(name="age (very weak)", 
                          regex=weak_regex, 
                          score=0.01)
    age_recognizer = PatternRecognizer(
        supported_entity="AGE",
        patterns = [age_pattern],
        name="AgeRecognizer",
        context=["month", "old", "turn", "age", "y/o"]
    )
    return age_recognizer


# Add a simple regex recognizer for zip codes
def get_zipcode_recognizer():
    # Very weak pattern - only triggers with strong context to avoid false positives
    very_weak_regex = r"\b\d{3,8}(?:[-\s]\d{2,4})?\b"
    
    # Weak pattern - US-style 5 digits or international with dash/space
    weak_regex = r"\b\d{5}(?:[-\s]\d{4})?\b"  # US: 12345 or 12345-6789
    
    # Medium pattern - with context words nearby
    med_regex = r"""
    (?:                                        # Non-capturing group for context
        (?:zip(?:\s?code)?|postal(?:\s?code)?|P\.?O\.?|mail)  # Context words
        [\s:,\-]{0,5}                          # Separator
    )
    (\d{3,8}(?:[-\s]\d{2,4})?)                 # ZIP code itself
    """
    
    # Strong pattern - explicit ZIP/postal code with number
    strong_regex = r"(?i)\b(?:zip(?:\s?code)?|postal\s?code)[:\s]+(\d{3,8}(?:[-\s]\d{2,4})?)\b"
    
    # Very weak: only with context boost (0.01 * context multiplier can reach >0.2)
    zipcode_pattern_very_weak = Pattern(name="zip code (very weak)", regex=very_weak_regex, score=0.01)
    # Weak: common US format
    zipcode_pattern_weak = Pattern(name="zip code (weak)", regex=weak_regex, score=0.25)
    # Medium: has zip/postal nearby
    zipcode_pattern_med = Pattern(name="zip code (medium)", regex=med_regex, score=0.6)
    # Strong: explicit labeling
    zipcode_pattern_strong = Pattern(name="zip code (strong)", regex=strong_regex, score=0.9)

    # Define the recognizer with the defined patterns
    zipcode_recognizer = PatternRecognizer(
        supported_entity="ZIP_CODE", 
        patterns=[zipcode_pattern_very_weak, zipcode_pattern_weak, zipcode_pattern_med, zipcode_pattern_strong], 
        name="ZipCodeRecognizer", 
        context=["zip", "zipcode", "postal", "code", "mail", "address", "sent", "ship"]
    )
    return zipcode_recognizer

# Add it to the Recognizer Registry
registry = RecognizerRegistry()
registry.load_predefined_recognizers(nlp_engine=nlp_engine)
registry.add_recognizer(get_titles_recognizer())
registry.add_recognizer(get_zipcode_recognizer())
registry.add_recognizer(get_location_deny_list())
registry.add_recognizer(get_age_recognizer())

# Remove unnecessary recognizers from presidio
unnecessary = ['NhsRecognizer', 'UkNinoRecognizer', 'SgFinRecognizer', 'AuAbnRecognizer', 
               'AuAcnRecognizer','AuTfnRecognizer', 'AuMedicareRecognizer', 'InPanRecognizer',
               'InAadhaarRecognizer', 'InVehicleRegistrationRecognizer', 'InPassportRecognizer', 
               'InVoterRecognizer', 'UsLicenseRecognizer', 'CryptoRecognizer']

for rec in unnecessary:
    registry.remove_recognizer(rec)

### 3.3 Configure the context mechanism
Configure the `LemmaContextAawareEnhancer` which uses surrounding words to increase confidence in detection

In [96]:
# Set up the context aware enhancer
context_enhancer = LemmaContextAwareEnhancer(context_prefix_count=10, 
                                             context_suffix_count=10)

### 3.4 Create the AnalyzerEngine object

In [97]:
# Set up the engine, loads the NLP module (spaCy model by default) 
# and other PII recognizers
analyzer_engine = AnalyzerEngine(nlp_engine=nlp_engine, 
                                 context_aware_enhancer=context_enhancer,
                                 registry=registry, 
                                 default_score_threshold=0.2)

# analyzer_engine = AnalyzerEngine()

pprint("Supported entities for English:")
pprint(analyzer_engine.get_supported_entities("en"), compact=True)

print("\nLoaded recognizers for English:")
pprint([rec.name for rec in analyzer_engine.registry.get_recognizers("en", all_fields=True)], compact=True)

print("\nLoaded Context Aware Enhancer:")
print(analyzer_engine.context_aware_enhancer.__class__.__name__)
pprint(json.dumps(analyzer_engine.context_aware_enhancer.__dict__), compact=True)


print("\nLoaded NER models:")
pprint(analyzer_engine.nlp_engine.models)

'Supported entities for English:'
['PHONE_NUMBER', 'EMAIL_ADDRESS', 'US_SSN', 'MEDICAL_LICENSE', 'LOCATION',
 'TITLE', 'US_BANK_NUMBER', 'US_PASSPORT', 'ID', 'AGE', 'DATE_TIME',
 'CREDIT_CARD', 'IBAN_CODE', 'PERSON', 'ORGANIZATION', 'US_ITIN', 'IP_ADDRESS',
 'URL', 'ZIP_CODE']

Loaded recognizers for English:
['CreditCardRecognizer', 'UsBankRecognizer', 'UsItinRecognizer',
 'UsPassportRecognizer', 'UsSsnRecognizer', 'DateRecognizer', 'EmailRecognizer',
 'IbanRecognizer', 'IpRecognizer', 'MedicalLicenseRecognizer',
 'PhoneRecognizer', 'UrlRecognizer', 'TransformersRecognizer',
 'TitlesRecognizer', 'ZipCodeRecognizer', 'LocationDenylist', 'AgeRecognizer']

Loaded Context Aware Enhancer:
LemmaContextAwareEnhancer
('{"context_similarity_factor": 0.35, "min_score_with_context_similarity": '
 '0.4, "context_prefix_count": 10, "context_suffix_count": 10}')

Loaded NER models:
[{'lang_code': 'en',
  'model_name': {'spacy': 'en_core_web_sm',
                 'transformers': 'StanfordAIMI/stanfo

In [98]:
# Test Analyzer
text="Yesterday in Mt. Sinai AP: Dana Silver, 79 years old female was complaining of stomach pain. Her ID is 154555"
res = analyzer_engine.analyze(text=text, 
                              language="en", 
                              return_decision_process=True)
for result in res:
    print(f"\nEntity: {result.entity_type}, Text: {text[result.start:result.end]}\n\nAnalysis explanation:")
    pprint(result.analysis_explanation)


Entity: LOCATION, Text: AP

Analysis explanation:
{'recognizer': 'LocationDenylist', 'pattern_name': 'deny_list', 'pattern': '(?:^|(?<=\\W))(APO|PSC|AA|Cyprus\\ \\(Greek\\)|ul|AE|DPO|AP|nan)(?:(?=\\W)|$)', 'original_score': 1.0, 'score': 1.0, 'textual_explanation': 'Detected by `LocationDenylist` using pattern `deny_list`', 'score_context_improvement': 0, 'supportive_context_word': '', 'validation_result': None, 'regex_flags': regex.I|M|S}

Entity: ORGANIZATION, Text: Mt. Sinai

Analysis explanation:
{'recognizer': 'TransformersRecognizer', 'pattern_name': None, 'pattern': None, 'original_score': np.float32(0.9806056), 'score': np.float32(0.9806056), 'textual_explanation': "Identified as ORGANIZATION by Transformers's Named Entity Recognition", 'score_context_improvement': 0, 'supportive_context_word': '', 'validation_result': None, 'regex_flags': None}

Entity: PERSON, Text: Dana Silver

Analysis explanation:
{'recognizer': 'TransformersRecognizer', 'pattern_name': None, 'pattern': N

## 4. Interactive Entity Mapping

Use `EntityMappingHelper` to automatically align dataset entities with model entities.

**Why entity mapping?** The dataset may use entity labels like `STREET_ADDRESS` or `GPE` (geopolitical entity), while the model outputs `LOCATION`. To properly evaluate the model, we need to map dataset entities to their corresponding model entities. The `EntityMappingHelper` automatically detects both sets of entities and suggests mappings, with the ability to manually adjust them.

For detailed information on entity mapping, see [notebook 6](6_Interactive_Entity_Mapping.ipynb).

In [99]:
# Create the helper - automatically detects entities and suggests mappings
helper = EntityMappingHelper(
    dataset=dataset,
    model=analyzer_engine,
    language="en"
)

# Review the suggested mapping
helper.review_mapping()

Dataset Entity,→ Model Entity,Samples,Confidence
✓ US_DRIVER_LICENSE,US_PASSPORT,5,0.82
✓ AGE,AGE,74,1.00
✓ CREDIT_CARD,CREDIT_CARD,136,1.00
✓ DATE_TIME,DATE_TIME,119,1.00
✓ DOMAIN_NAME,URL,37,1.00
✓ EMAIL_ADDRESS,EMAIL_ADDRESS,49,1.00
✓ GPE,LOCATION,325,1.00
✓ IBAN_CODE,IBAN_CODE,21,1.00
✓ IP_ADDRESS,IP_ADDRESS,14,1.00
✓ NRP,ORGANIZATION,55,1.00


In [100]:
# Manually update the mapping - align dataset entities to model entities
helper.set_mapping("GPE", "LOCATION")  # Geopolitical entities (cities, countries) → Locations
helper.set_mapping("NRP", "ORGANIZATION")  # Nationalities/religious/political groups → Organizations
helper.set_mapping("DOMAIN_NAME", "URL")  # Domain names are URLs
helper.set_mapping("US_DRIVER_LICENSE", "ID")  # Driver's licenses are IDs
helper.set_mapping("EMAIL", "EMAIL_ADDRESS")  # Normalize email entity name
helper.set_mapping("ZIP_CODE","ZIP_CODE")  # Keep zip codes as-is, do not map to LOCATION
helper.set_mapping("STREET_ADDRESS", "LOCATION")  # Addresses are locations


helper.review_mapping()

print("✓ Using suggested mapping")

✓ Mapping set: GPE → LOCATION
✓ Mapping set: NRP → ORGANIZATION
✓ Mapping set: DOMAIN_NAME → URL
✓ Mapping set: US_DRIVER_LICENSE → ID
⚠️  Warning: 'EMAIL' not found in dataset entities
✓ Mapping set: ZIP_CODE → ZIP_CODE
✓ Mapping set: STREET_ADDRESS → LOCATION


Dataset Entity,→ Model Entity,Samples,Confidence
✓ AGE,AGE,74,1.00
✓ CREDIT_CARD,CREDIT_CARD,136,1.00
✓ DATE_TIME,DATE_TIME,119,1.00
✓ EMAIL_ADDRESS,EMAIL_ADDRESS,49,1.00
✓ IBAN_CODE,IBAN_CODE,21,1.00
✓ IP_ADDRESS,IP_ADDRESS,14,1.00
✓ ORGANIZATION,ORGANIZATION,199,1.00
✓ PERSON,PERSON,637,1.00
✓ PHONE_NUMBER,PHONE_NUMBER,64,1.00
✓ US_SSN,US_SSN,16,1.00


✓ Using suggested mapping


In [101]:
# Get final mappings and filtered dataset
entities_mapping = helper.get_mapping()
model_entities_to_use = helper.get_model_entities_to_use()
dataset = helper.get_filtered_dataset()

print("=== Final Configuration ===")
print(f"\nDataset → Model mapping: {len(entities_mapping)} entities")
pprint(entities_mapping, compact=True)

print(f"\nModel entities to evaluate: {sorted(model_entities_to_use)}")
print(f"Filtered dataset: {len(dataset)} samples")

print("\n✓ Ready for evaluation")

=== Final Configuration ===

Dataset → Model mapping: 17 entities
{'AGE': 'AGE',
 'CREDIT_CARD': 'CREDIT_CARD',
 'DATE_TIME': 'DATE_TIME',
 'DOMAIN_NAME': 'URL',
 'EMAIL_ADDRESS': 'EMAIL_ADDRESS',
 'GPE': 'LOCATION',
 'IBAN_CODE': 'IBAN_CODE',
 'IP_ADDRESS': 'IP_ADDRESS',
 'NRP': 'ORGANIZATION',
 'ORGANIZATION': 'ORGANIZATION',
 'PERSON': 'PERSON',
 'PHONE_NUMBER': 'PHONE_NUMBER',
 'STREET_ADDRESS': 'LOCATION',
 'TITLE': 'TITLE',
 'US_DRIVER_LICENSE': 'ID',
 'US_SSN': 'US_SSN',
 'ZIP_CODE': 'ZIP_CODE'}

Model entities to evaluate: ['AGE', 'CREDIT_CARD', 'DATE_TIME', 'EMAIL_ADDRESS', 'IBAN_CODE', 'ID', 'IP_ADDRESS', 'LOCATION', 'ORGANIZATION', 'PERSON', 'PHONE_NUMBER', 'TITLE', 'URL', 'US_SSN', 'ZIP_CODE']
Filtered dataset: 1387 samples

✓ Ready for evaluation


## 5. Set up the Evaluator object

In [102]:
# Set up the experiment tracker
experiment = get_experiment_tracker()

# Wrap the analyzer
wrapped_analyzer = PresidioAnalyzerWrapper(
    analyzer_engine=analyzer_engine,
    score_threshold=analyzer_engine.default_score_threshold,
    language="en"
)

# Create the evaluator with entity mapping
evaluator = SpanEvaluator(
    model=wrapped_analyzer,
    entity_mapping=entities_mapping,
    iou_threshold=0.7
)

# Track experiment parameters
params = {"dataset_name": dataset_name, "model_name": evaluator.model.name}
params.update(evaluator.model.to_log())
experiment.log_parameters(params)
experiment.log_dataset_hash(dataset)
experiment.log_parameter("entity_mappings", json.dumps(entities_mapping))

--------
Entities supported by this Presidio Analyzer instance:
PHONE_NUMBER, EMAIL_ADDRESS, US_SSN, MEDICAL_LICENSE, LOCATION, TITLE, US_BANK_NUMBER, US_PASSPORT, ID, AGE, DATE_TIME, CREDIT_CARD, IBAN_CODE, PERSON, ORGANIZATION, US_ITIN, IP_ADDRESS, URL, ZIP_CODE


## 6. Run experiment

In [103]:
%%time

## Run experiment

evaluation_results = evaluator.evaluate_all(dataset)
results = evaluator.calculate_score(evaluation_results)

# Track experiment results
experiment.log_metrics(results.to_log())
entities, confmatrix = results.to_confusion_matrix()
experiment.log_confusion_matrix(matrix=confmatrix, 
                                labels=entities)

# end experiment
experiment.end()

# Note that the experiment params and metrics are saved locally

Using entity mapping for comparison: {'GPE': 'LOCATION', 'ORGANIZATION': 'ORGANIZATION', 'AGE': 'AGE', 'EMAIL_ADDRESS': 'EMAIL_ADDRESS', 'DATE_TIME': 'DATE_TIME', 'CREDIT_CARD': 'CREDIT_CARD', 'NRP': 'ORGANIZATION', 'US_DRIVER_LICENSE': 'ID', 'STREET_ADDRESS': 'LOCATION', 'PHONE_NUMBER': 'PHONE_NUMBER', 'IBAN_CODE': 'IBAN_CODE', 'TITLE': 'TITLE', 'DOMAIN_NAME': 'URL', 'PERSON': 'PERSON', 'IP_ADDRESS': 'IP_ADDRESS', 'ZIP_CODE': 'ZIP_CODE', 'US_SSN': 'US_SSN'}
Running model PresidioAnalyzerWrapper on dataset...
Finished running model on dataset
saving experiment data to /Users/omrimendels/Documents/github/presidio-research/notebooks/experiment_20260228-122107.json
CPU times: user 40.3 s, sys: 23.9 s, total: 1min 4s
Wall time: 49.5 s


## 7. Evaluate results

In [104]:
# Plot output
plotter = Plotter(results=results, 
                  model_name = evaluator.model.name, 
                  save_as='html',
                  beta = 2)
output_folder = Path(Path.cwd().parent, "plotter_output")
plotter.plot_scores(output_folder=output_folder)

In [105]:
pprint({"PII F":results.pii_f, "PII recall": results.pii_recall, "PII precision": results.pii_precision})

{'PII F': 0.8439860546194073,
 'PII precision': 0.8723723723723724,
 'PII recall': 0.8371757925072046}


## 8. Error analysis

The `ModelError` class groups similar errors together, finding all the false positive and false negative predictions where the model's behavior is consistent. 

This can help in detecting patterns in the model's performance, such as:
- Consistent labeling of specific words or patterns  
- Systematic overprediction or underprediction

**Note:** All errors are displayed using the **mapped model entity types** (e.g., `LOCATION`, `PERSON`) rather than the original dataset entity types (e.g., `STREET_ADDRESS`, `GPE`). This ensures consistency with how the model was evaluated.

In [106]:
plotter.plot_confusion_matrix(entities=entities, confmatrix=confmatrix, output_folder=output_folder)

In [107]:
plotter.plot_most_common_tokens(output_folder=output_folder)

### 8a. False positives
#### Most common false positive tokens:

In [108]:
ModelError.most_common_fp_tokens(results.model_errors)

Most common false positive tokens:
[('nan', 20),
 ('psc', 17),
 ('estonia', 16),
 ('card', 14),
 ('ul', 14),
 ('iban', 13),
 ('greenlander', 13),
 ('baroque pop', 10),
 ('8 16', 10),
 ('norway', 10)]
---------------
Example sentence with each FP token:
	- nan (`nan` pred as LOCATION)
	- PSC (`psc` pred as LOCATION)
	- Estonia (`estonia` pred as PERSON)
	- card (`card` pred as ORGANIZATION)
	- ul (`ul` pred as LOCATION)
	- my iban (`iban` pred as PERSON)
	- Greenlander (`greenlander` pred as ORGANIZATION)
	- baroque pop (`baroque pop` pred as PERSON)
	- 8 16 (`8 16` pred as AGE)
	- Norway (`norway` pred as ORGANIZATION)


[('nan', 20),
 ('psc', 17),
 ('estonia', 16),
 ('card', 14),
 ('ul', 14),
 ('iban', 13),
 ('greenlander', 13),
 ('baroque pop', 10),
 ('8 16', 10),
 ('norway', 10)]

#### More FP analysis

In [109]:
fps_df = ModelError.get_fps_dataframe(results.model_errors, entity="AGE")
fps_df[["full_text", "token", "annotation", "prediction"]].head(20)

,full_text,token,annotation,prediction
0,501(c)3,501(c)3,O,AGE
1,8 16,8 16,O,AGE
2,8 16,8 16,O,AGE
3,8 16,8 16,O,AGE
4,49,49,O,AGE
5,20,20,O,AGE
6,501(c)3,501(c)3,O,AGE
7,501(c)3,501(c)3,O,AGE
8,13,13,O,AGE
9,501(c)3,501(c)3,O,AGE


### 8b. False negatives (FN)

#### Most common false negative examples + a few samples with FN

In [110]:
ModelError.most_common_fn_tokens(results.model_errors, n=15)

Most common false negative tokens:
[('estonia', 17),
 ('norway', 13),
 ('greenlander', 12),
 ('czech republic', 9),
 ('iceland', 8),
 ('sweden', 8),
 ('canada', 7),
 ('france', 7),
 ('united states', 7),
 ('hungary', 7),
 ('tunisia', 6),
 ('uruguay', 6),
 ('finland', 5),
 ('cyprus', 5),
 ('american', 5)]
---------------
Example sentence with each FN token:
	- Estonia (`estonia` annotated as LOCATION)
	- Norway (`norway` annotated as LOCATION)
	- Greenlander (`greenlander` annotated as LOCATION)
	- Czech Republic (`czech republic` annotated as LOCATION)
	- Iceland (`iceland` annotated as LOCATION)
	- Sweden (`sweden` annotated as LOCATION)
	- Canada (`canada` annotated as LOCATION)
	- France (`france` annotated as LOCATION)
	- United States (`united states` annotated as LOCATION)
	- Hungary (`hungary` annotated as LOCATION)
	- Tunisia (`tunisia` annotated as LOCATION)
	- Uruguay (`uruguay` annotated as LOCATION)
	- Finland (`finland` annotated as LOCATION)
	- Cyprus (`cyprus` annotated 

[('estonia', 17),
 ('norway', 13),
 ('greenlander', 12),
 ('czech republic', 9),
 ('iceland', 8),
 ('sweden', 8),
 ('canada', 7),
 ('france', 7),
 ('united states', 7),
 ('hungary', 7),
 ('tunisia', 6),
 ('uruguay', 6),
 ('finland', 5),
 ('cyprus', 5),
 ('american', 5)]

#### More FN analysis

In [111]:
fns_df = ModelError.get_fns_dataframe(results.model_errors, entity="PERSON")

In [112]:
fns_df[["full_text", "token", "annotation", "prediction"]].head(20)

,full_text,token,annotation,prediction
0,Christiansen,christiansen,PERSON,O
1,Sara Schwarz,sara schwarz,PERSON,O
2,Jensen,jensen,PERSON,O
3,Eklund Michael Hodge,eklund michael hodge,PERSON,O
4,Kaczmarek,kaczmarek,PERSON,O
5,Bonifacy Kaczmarek,bonifacy kaczmarek,PERSON,O
6,Mette,mette,PERSON,O
7,Katrine,katrine,PERSON,O
8,Ravil G Yefimov,ravil g yefimov,PERSON,O
9,Kevin Alma,kevin alma,PERSON,O
